# Stigma Classifier — PEFT (LoRA) Fine-Tuning of Llama / Qwen — 5-Fold CV

Swaps the RoBERTa full fine-tune for 4-bit QLoRA fine-tuning of a decoder-only LLM
(Llama-3.1-8B or Qwen2.5-7B) used as a sequence classifier.

**Runtime → Change runtime type → A100 GPU recommended.** A T4 (16GB) can work with
the conservative batch settings below but will be slow and may OOM depending on
sequence lengths.

If using a Llama checkpoint, you need a Hugging Face account with access to the
gated `meta-llama` repo, and a token with read access.

In [ ]:
!pip install -q -U transformers datasets scikit-learn peft bitsandbytes accelerate

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload golden_all_batches.xlsx

In [ ]:
# ── HF auth (only needed for gated repos, e.g. meta-llama) ─────────────────────
from huggingface_hub import login
login()  # paste a token with read access when prompted

In [ ]:
import gc
import torch
import pandas as pd
import random
import numpy as np
import os
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, cohen_kappa_score
from sklearn.model_selection import StratifiedKFold
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('social_stigma')

print('GPU available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('bf16 supported:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [ ]:
# ── Base model choice ───────────────────────────────────────────────────────
# Swap MODEL_CHOICE to switch backbones. Instruct checkpoints, since the input
# is formatted as an instruction below and classification is done on top of
# the resulting representation (not the base completion distribution).
MODEL_CHOICE = 'qwen'  # 'qwen' or 'llama'

MODEL_IDS = {
    'qwen':  'Qwen/Qwen2.5-7B-Instruct',
    'llama': 'meta-llama/Llama-3.1-8B-Instruct',
}
model_id = MODEL_IDS[MODEL_CHOICE]
print('Using base model:', model_id)

In [ ]:
# ── Labels ────────────────────────────────────────────────────────────────────
label_dict      = {'no': 0, 'yes': 1}
id2label_custom = {v: k for k, v in label_dict.items()}

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_excel('golden_all_batches__.xlsx')[['text', 'stigma_present']].rename(columns={'stigma_present': 'label'})

df['label'] = df['label'].str.strip().str.lower()  # normalise capitalisation
df['label_id'] = df['label'].map(label_dict)

print('Total samples:', len(df))
print(df['label'].value_counts())

In [ ]:
# ── Tokenizer + instruction formatting ─────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

SYSTEM_PROMPT = (
    "You are a classifier that determines whether a piece of text expresses or "
    "conveys stigma toward people with mental health conditions. Read the text "
    "and judge whether stigmatizing framing, language, or attitudes are present."
)

def format_prompt(text):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f'Text: """{text}"""'},
    ]
    # add_generation_prompt=True appends the assistant turn header, so the
    # classification head pools a representation conditioned on "about to answer"
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def encode_text(examples):
    prompts = [format_prompt(t) for t in examples['text']]
    return tokenizer(prompts, max_length=768, padding='max_length', truncation=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    pos = label_dict['yes']
    return {
        'f1_binary':        f1_score(labels, predictions, pos_label=pos, average='binary'),
        'precision_binary': precision_score(labels, predictions, pos_label=pos, average='binary'),
        'recall_binary':    recall_score(labels, predictions, pos_label=pos, average='binary'),
        'f1_macro':         f1_score(labels, predictions, average='macro'),
        'precision_macro':  precision_score(labels, predictions, average='macro'),
        'recall_macro':     recall_score(labels, predictions, average='macro'),
        'accuracy':         accuracy_score(labels, predictions),
        'cohen_kappa':      cohen_kappa_score(labels, predictions),
    }

In [ ]:
# ── QLoRA setup ───────────────────────────────────────────────────────────────
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = 'nf4',
    bnb_4bit_compute_dtype    = compute_dtype,
    bnb_4bit_use_double_quant = True,
)

lora_config = LoraConfig(
    task_type        = TaskType.SEQ_CLS,
    r                = 16,
    lora_alpha       = 32,
    lora_dropout     = 0.05,
    bias             = 'none',
    target_modules   = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                         'gate_proj', 'up_proj', 'down_proj'],
)

def build_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels          = 2,
        quantization_config = bnb_config,
        device_map           = 'auto',
        torch_dtype          = compute_dtype,
    )
    model.config.id2label = id2label_custom
    model.config.label2id = label_dict
    model.config.pad_token_id = tokenizer.pad_token_id

    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model

In [ ]:
# ── 5-Fold CV ─────────────────────────────────────────────────────────────────
skf     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X       = df['text'].values
y       = df['label_id'].values

fold_results  = []
all_preds     = np.zeros(len(df), dtype=int)
all_true      = np.zeros(len(df), dtype=int)

for fold, (train_val_idx, test_idx) in enumerate(skf.split(X, y)):
    print(f'\n=== Fold {fold+1}/5 ===')

    # Split train_val into train (80%) + val (20%), stratified
    X_train_val = X[train_val_idx]
    y_train_val = y[train_val_idx]
    inner_skf   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    train_sub_idx, val_sub_idx = next(inner_skf.split(X_train_val, y_train_val))
    train_idx_final = train_val_idx[train_sub_idx]
    val_idx_final   = train_val_idx[val_sub_idx]

    train_df_fold = df.iloc[train_idx_final].reset_index(drop=True)
    val_df_fold   = df.iloc[val_idx_final].reset_index(drop=True)
    test_df_fold  = df.iloc[test_idx].reset_index(drop=True)

    def to_ds(subset):
        ds = Dataset.from_pandas(subset[['text', 'label_id']].rename(columns={'label_id': 'label'}))
        return ds.map(encode_text, batched=True)

    train_ds = to_ds(train_df_fold)
    val_ds   = to_ds(val_df_fold)
    test_ds  = to_ds(test_df_fold)

    # Fresh quantized base model + fresh LoRA adapters per fold
    model = build_model()

    train_arg = TrainingArguments(
        output_dir                  = f'./results/fold_{fold+1}',
        num_train_epochs            = 3,
        per_device_train_batch_size = 2,
        per_device_eval_batch_size  = 4,
        gradient_accumulation_steps = 16,
        gradient_checkpointing      = True,
        seed                        = 42,
        learning_rate               = 2e-4,
        bf16                        = torch.cuda.is_bf16_supported(),
        fp16                        = not torch.cuda.is_bf16_supported(),
        optim                       = 'paged_adamw_8bit',
        eval_steps                  = 50,
        save_strategy               = 'steps',
        save_steps                  = 50,
        eval_strategy               = 'steps',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'f1_binary',
        report_to                   = 'none',
    )

    trainer = Trainer(
        model           = model,
        args            = train_arg,
        train_dataset   = train_ds,
        eval_dataset    = val_ds,   # val split, not test
        compute_metrics = compute_metrics,
    )

    trainer.train()

    # Predict on held-out test split
    preds        = trainer.predict(test_ds)
    pred_ids     = preds.predictions.argmax(axis=1)
    true_ids     = test_ds['label']

    all_preds[test_idx] = pred_ids
    all_true[test_idx]  = true_ids

    pos = label_dict['yes']
    fold_metrics = {
        'fold':             fold + 1,
        'f1_binary':        f1_score(true_ids, pred_ids, pos_label=pos, average='binary'),
        'precision_binary': precision_score(true_ids, pred_ids, pos_label=pos, average='binary'),
        'recall_binary':    recall_score(true_ids, pred_ids, pos_label=pos, average='binary'),
        'f1_macro':         f1_score(true_ids, pred_ids, average='macro'),
        'precision_macro':  precision_score(true_ids, pred_ids, average='macro'),
        'recall_macro':     recall_score(true_ids, pred_ids, average='macro'),
        'accuracy':         accuracy_score(true_ids, pred_ids),
        'cohen_kappa':      cohen_kappa_score(true_ids, pred_ids),
    }
    fold_results.append(fold_metrics)
    print(fold_metrics)

    # Free the quantized model + adapters before the next fold
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# ── Aggregate results ─────────────────────────────────────────────────────────
results_df = pd.DataFrame(fold_results)
print('\n=== Per-fold results ===')
print(results_df.to_string(index=False))

metric_cols = [c for c in results_df.columns if c != 'fold']
print('\n=== Mean ± Std across folds ===')
for col in metric_cols:
    print(f"{col:20s}: {results_df[col].mean():.4f} ± {results_df[col].std():.4f}")

pos = label_dict['yes']
print('\n=== Overall (all folds combined) ===')
print(f"Accuracy:            {accuracy_score(all_true, all_preds):.4f}")
print(f"Precision (binary):  {precision_score(all_true, all_preds, pos_label=pos, average='binary'):.4f}")
print(f"Recall    (binary):  {recall_score(all_true, all_preds, pos_label=pos, average='binary'):.4f}")
print(f"F1        (binary):  {f1_score(all_true, all_preds, pos_label=pos, average='binary'):.4f}")
print(f"Precision (macro):   {precision_score(all_true, all_preds, average='macro'):.4f}")
print(f"Recall    (macro):   {recall_score(all_true, all_preds, average='macro'):.4f}")
print(f"F1        (macro):   {f1_score(all_true, all_preds, average='macro'):.4f}")
print(f"Cohen Kappa:         {cohen_kappa_score(all_true, all_preds):.4f}")

In [ ]:
# ── Save predictions and download ─────────────────────────────────────────────
output_df = df[['text', 'label']].copy()
output_df['pred_label'] = [id2label_custom[p] for p in all_preds]
output_df.to_csv('roberta_kfold_predictions.csv', index=False, encoding='utf-8')

results_df.to_csv('roberta_kfold_metrics.csv', index=False)

files.download('roberta_kfold_predictions.csv')
files.download('roberta_kfold_metrics.csv')